# SilverAI - Data Collection and Understanding

## Objective
The objective of this notebook is to load and understand historical
silver market data before applying Machine Learning.

## Datasets
1. SI_1m_1098d.csv - One-minute silver futures data.
2. SI_daily_roll-adjusted.csv - Daily silver futures data.

## Tasks
- Load historical datasets.
- Understand dataset structure.
- Identify OHLCV columns.
- Inspect missing values and duplicates.
- Check timestamp consistency.
- Analyze data quality.
- Generate initial statistical summaries.

## Asset
COMEX Silver Futures (SI)

Import libraries

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
import logging
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Libraries imported successfully.")

Libraries imported successfully.


Configure project paths and logging

In [16]:
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True
)

logger = logging.getLogger("SilverAI")

print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)

if not DATA_DIR.exists():
    raise FileNotFoundError("The data directory does not exist.")

logger.info("SilverAI data collection notebook initialized.")

2026-11-08 11:09:03,934 | INFO | SilverAI data collection notebook initialized.


Project directory: c:\Users\ravik\Desktop\SilverAI
Data directory: c:\Users\ravik\Desktop\SilverAI\data


Verify available datasets

In [17]:
files = list(DATA_DIR.glob("*.csv"))

print("Available CSV files:\n")

for file in files:
    print(f"- {file.name} | {file.stat().st_size / (1024**2):.2f} MB")

if not files:
    logger.warning("No CSV datasets found in the data directory.")

Available CSV files:

- SI_1m_1098d.csv | 45.19 MB
- SI_daily_roll-adjusted.csv | 0.41 MB


Load our datasets

In [18]:
INTRADAY_FILE = DATA_DIR / "SI_1m_1098d.csv"
DAILY_FILE = DATA_DIR / "SI_daily_roll-adjusted.csv"

def load_dataset(path):
    if not path.exists():
        logger.warning(f"Dataset not found: {path.name}")
        return None

    df = pd.read_csv(path)

    logger.info(
        f"Loaded {path.name} | "
        f"Rows: {df.shape[0]} | Columns: {df.shape[1]}"
    )

    return df


intraday_df = load_dataset(INTRADAY_FILE)
daily_df = load_dataset(DAILY_FILE)

2026-11-08 11:09:05,422 | INFO | Loaded SI_1m_1098d.csv | Rows: 1031365 | Columns: 6
2026-11-08 11:09:05,517 | INFO | Loaded SI_daily_roll-adjusted.csv | Rows: 7775 | Columns: 6


First look at the datasets

In [19]:
if intraday_df is not None:
    print("INTRADAY DATA")
    display(intraday_df.head())

if daily_df is not None:
    print("DAILY DATA")
    display(daily_df.head())

INTRADAY DATA


,datetime_et,open,high,low,close,volume
0,2023-11-01 02:09,22.745,22.755,22.745,22.755,21
1,2023-11-01 02:10,22.750,22.755,22.750,22.755,23
2,2023-11-01 02:11,22.760,22.760,22.745,22.755,70
3,2023-11-01 02:12,22.755,22.765,22.755,22.760,20
4,2023-11-01 02:13,22.765,22.770,22.760,22.770,31


DAILY DATA


,date,open,high,low,close,volume
0,1995-11-29,9.269814,9.296124,9.138265,9.150543,26462
1,1995-11-30,9.164575,9.252274,9.059336,9.238242,16144
2,1995-12-01,9.252274,9.278584,9.182115,9.194393,9022
3,1995-12-04,9.208424,9.225964,9.147035,9.185623,7118
4,1995-12-05,9.190885,9.278584,9.164575,9.255782,10728


Dataset dimensions and structure

In [20]:
def dataset_overview(df, name):
    if df is None:
        return

    print(f"\n{'=' * 50}")
    print(name)
    print("=" * 50)

    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])

    print("\nColumn names:")
    print(df.columns.tolist())

    print("\nData types:")
    display(df.dtypes.to_frame("Data Type"))

    print("\nDataset information:")
    df.info()


dataset_overview(intraday_df, "INTRADAY DATA")
dataset_overview(daily_df, "DAILY DATA")


INTRADAY DATA
Rows: 1031365
Columns: 6

Column names:
['datetime_et', 'open', 'high', 'low', 'close', 'volume']

Data types:


,Data Type
datetime_et,object
open,float64
high,float64
low,float64
close,float64
volume,int64



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1031365 entries, 0 to 1031364
Data columns (total 6 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   datetime_et  1031365 non-null  object 
 1   open         1031365 non-null  float64
 2   high         1031365 non-null  float64
 3   low          1031365 non-null  float64
 4   close        1031365 non-null  float64
 5   volume       1031365 non-null  int64  
dtypes: float64(4), int64(1), object(1)
memory usage: 47.2+ MB

DAILY DATA
Rows: 7775
Columns: 6

Column names:
['date', 'open', 'high', 'low', 'close', 'volume']

Data types:


,Data Type
date,object
open,float64
high,float64
low,float64
close,float64
volume,int64



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7775 entries, 0 to 7774
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    7775 non-null   object 
 1   open    7775 non-null   float64
 2   high    7775 non-null   float64
 3   low     7775 non-null   float64
 4   close   7775 non-null   float64
 5   volume  7775 non-null   int64  
dtypes: float64(4), int64(1), object(1)
memory usage: 364.6+ KB


Identify OHLCV columns

In [21]:
ALIASES = {
    "timestamp": [
        "timestamp",
        "datetime",
        "datetime_et",
        "date",
        "time",
        "open_time",
        "date_time"
    ],
    "open": ["open", "o"],
    "high": ["high", "h"],
    "low": ["low", "l"],
    "close": ["close", "adj close", "adj_close", "c"],
    "volume": ["volume", "vol", "v"]
}


def identify_columns(df):
    if df is None:
        return {}

    result = {}

    for standard_name, aliases in ALIASES.items():
        for col in df.columns:
            normalized = str(col).strip().lower()

            if normalized in aliases:
                result[standard_name] = col
                break

    return result


intraday_columns = identify_columns(intraday_df)
daily_columns = identify_columns(daily_df)

print("Intraday column mapping:")
print(intraday_columns)

print("\nDaily column mapping:")
print(daily_columns)

Intraday column mapping:
{'timestamp': 'datetime_et', 'open': 'open', 'high': 'high', 'low': 'low', 'close': 'close', 'volume': 'volume'}

Daily column mapping:
{'timestamp': 'date', 'open': 'open', 'high': 'high', 'low': 'low', 'close': 'close', 'volume': 'volume'}


Missing values

In [22]:
def missing_value_report(df, name):
    if df is None:
        return

    report = pd.DataFrame({
        "Missing Count": df.isnull().sum(),
        "Missing Percentage": df.isnull().mean() * 100
    })

    print(f"\n{name} - Missing Value Report")
    display(report)


missing_value_report(intraday_df, "INTRADAY")
missing_value_report(daily_df, "DAILY")


INTRADAY - Missing Value Report


,Missing Count,Missing Percentage
datetime_et,0,0.0
open,0,0.0
high,0,0.0
low,0,0.0
close,0,0.0
volume,0,0.0



DAILY - Missing Value Report


,Missing Count,Missing Percentage
date,0,0.0
open,0,0.0
high,0,0.0
low,0,0.0
close,0,0.0
volume,0,0.0


Duplicate records

In [23]:
def duplicate_report(df, name):
    if df is None:
        return

    duplicates = df.duplicated().sum()

    print(f"{name}")
    print("Duplicate rows:", duplicates)
    print("Unique rows:", len(df) - duplicates)


duplicate_report(intraday_df, "INTRADAY")
duplicate_report(daily_df, "DAILY")

INTRADAY
Duplicate rows: 0
Unique rows: 1031365
DAILY
Duplicate rows: 0
Unique rows: 7775


Statistical summary

In [24]:
def numerical_summary(df, name):
    if df is None:
        return

    print(f"\n{name} - Numerical Summary")

    display(df.describe().T)


numerical_summary(intraday_df, "INTRADAY")
numerical_summary(daily_df, "DAILY")


INTRADAY - Numerical Summary


,count,mean,std,min,25%,50%,75%,max
open,1031365.0,44.251848,19.951841,21.930,29.845,33.815,59.865,121.710
high,1031365.0,44.269914,19.969193,21.945,29.850,33.825,59.885,121.785
low,1031365.0,44.233317,19.933730,21.925,29.835,33.810,59.840,121.595
close,1031365.0,44.251863,19.951818,21.930,29.845,33.815,59.865,121.695
volume,1031365.0,44.445847,76.950754,0.000,8.000,20.000,49.000,3840.000



DAILY - Numerical Summary


,count,mean,std,min,25%,50%,75%,max
open,7775.0,22.933148,14.502848,6.652152,9.918789,21.691439,28.913702,122.764226
high,7775.0,23.281462,14.870663,6.701795,10.059655,21.951463,29.313761,128.195852
low,7775.0,22.555339,14.095317,6.643878,9.764738,21.424573,28.496741,116.211537
close,7775.0,22.923957,14.485394,6.665390,9.910019,21.687334,28.928444,121.584215
volume,7775.0,41489.071254,33160.348302,0.000000,15117.500000,34671.000000,59205.500000,355275.000000


Timestamp analysis

In [25]:
def timestamp_report(df, mapping, name):
    if df is None or "timestamp" not in mapping:
        print(f"{name}: Timestamp column not identified.")
        return

    col = mapping["timestamp"]

    timestamps = pd.to_datetime(
        df[col],
        errors="coerce"
    )

    if timestamps.dt.tz is None:
        timestamps = timestamps.dt.tz_localize(
            "America/New_York",
            ambiguous="NaT",
            nonexistent="NaT"
        )

    timestamps = timestamps.dt.tz_convert("UTC")

    valid = timestamps.dropna().sort_values()

    print(f"\n{name} TIMESTAMP REPORT")
    print("Invalid timestamps:", timestamps.isna().sum())

    if len(valid) == 0:
        print("No valid timestamps found.")
        return

    print("First timestamp:", valid.iloc[0])
    print("Last timestamp:", valid.iloc[-1])
    print("Total valid timestamps:", len(valid))

    differences = valid.diff().dropna()

    print("\nMost frequent intervals:")
    display(
        differences.value_counts().head(10).to_frame("Frequency")
    )

timestamp_report(
    intraday_df, intraday_columns, "INTRADAY"
)

timestamp_report(
    daily_df, daily_columns, "DAILY"
)


INTRADAY TIMESTAMP REPORT
Invalid timestamps: 0
First timestamp: 2023-11-01 06:09:00+00:00
Last timestamp: 2026-11-03 07:08:00+00:00
Total valid timestamps: 1031365

Most frequent intervals:


,Frequency
datetime_et,
0 days 00:01:00,1004779
0 days 00:02:00,20980
0 days 00:03:00,3439
0 days 00:04:00,894
0 days 01:01:00,593
0 days 00:05:00,272
2 days 01:01:00,138
0 days 00:06:00,100
0 days 00:07:00,37



DAILY TIMESTAMP REPORT
Invalid timestamps: 0
First timestamp: 1995-11-29 05:00:00+00:00
Last timestamp: 2026-11-02 05:00:00+00:00
Total valid timestamps: 7775

Most frequent intervals:


,Frequency
date,
1 days 00:00:00,6102
3 days 00:00:00,1338
4 days 00:00:00,198
2 days 00:00:00,57
3 days 01:00:00,30
2 days 23:00:00,29
5 days 00:00:00,17
3 days 23:00:00,2
4 days 01:00:00,1


OHLC data validation

In [26]:
def ohlc_validation(df, mapping, name):
    if df is None:
        return

    required = ["open", "high", "low", "close"]

    if not all(col in mapping for col in required):
        print(f"{name}: Missing OHLC columns. Validation skipped.")
        return

    o = pd.to_numeric(df[mapping["open"]], errors="coerce")
    h = pd.to_numeric(df[mapping["high"]], errors="coerce")
    l = pd.to_numeric(df[mapping["low"]], errors="coerce")
    c = pd.to_numeric(df[mapping["close"]], errors="coerce")

    invalid = (
        (h < l) |
        (h < o) |
        (h < c) |
        (l > o) |
        (l > c) |
        (o <= 0) |
        (h <= 0) |
        (l <= 0) |
        (c <= 0)
    )

    print(f"\n{name} OHLC VALIDATION")
    print("Invalid OHLC rows:", invalid.sum())
    print("Valid OHLC rows:", (~invalid).sum())

    if "volume" in mapping:
        volume = pd.to_numeric(
            df[mapping["volume"]], errors="coerce"
        )

        print("Invalid volume values:", volume.isna().sum())
        print("Negative volume values:", (volume < 0).sum())


ohlc_validation(intraday_df, intraday_columns, "INTRADAY")
ohlc_validation(daily_df, daily_columns, "DAILY")


INTRADAY OHLC VALIDATION
Invalid OHLC rows: 0
Valid OHLC rows: 1031365
Invalid volume values: 0
Negative volume values: 0

DAILY OHLC VALIDATION
Invalid OHLC rows: 0
Valid OHLC rows: 7775
Invalid volume values: 0
Negative volume values: 0


Initial conclusion

In [27]:
print("=" * 60)
print("SILVERAI - DATA COLLECTION SUMMARY")
print("=" * 60)

for name, df in [
    ("Intraday", intraday_df),
    ("Daily", daily_df)
]:
    if df is None:
        print(f"{name}: Dataset unavailable.")
        continue

    print(f"\n{name} Dataset")
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Missing values:", df.isnull().sum().sum())
    print("Duplicate rows:", df.duplicated().sum())

print("\nData collection and initial inspection completed.")
logger.info("Data inspection completed.")

SILVERAI - DATA COLLECTION SUMMARY

Intraday Dataset
Rows: 1031365
Columns: 6
Missing values: 0


2026-11-08 11:09:17,259 | INFO | Data inspection completed.


Duplicate rows: 0

Daily Dataset
Rows: 7775
Columns: 6
Missing values: 0
Duplicate rows: 0

Data collection and initial inspection completed.


In [28]:
print("INTRADAY COLUMN NAMES:")
print(intraday_df.columns.tolist())

print("\nFIRST 10 ROWS:")
display(intraday_df.head(10))

print("\nDATA TYPES:")
print(intraday_df.dtypes)

INTRADAY COLUMN NAMES:
['datetime_et', 'open', 'high', 'low', 'close', 'volume']

FIRST 10 ROWS:


,datetime_et,open,high,low,close,volume
0,2023-11-01 02:09,22.745,22.755,22.745,22.755,21
1,2023-11-01 02:10,22.750,22.755,22.750,22.755,23
2,2023-11-01 02:11,22.760,22.760,22.745,22.755,70
3,2023-11-01 02:12,22.755,22.765,22.755,22.760,20
4,2023-11-01 02:13,22.765,22.770,22.760,22.770,31
5,2023-11-01 02:14,22.770,22.790,22.770,22.770,85
6,2023-11-01 02:15,22.770,22.770,22.765,22.765,12
7,2023-11-01 02:16,22.770,22.785,22.770,22.780,22
8,2023-11-01 02:17,22.780,22.785,22.780,22.780,5
9,2023-11-01 02:18,22.785,22.790,22.780,22.785,25



DATA TYPES:
datetime_et     object
open           float64
high           float64
low            float64
close          float64
volume           int64
dtype: object
